In [1]:
from google.adk.agents.llm_agent import Agent

In [2]:
# Cell 3
import google.auth
from google.adk.telemetry import google_cloud
from google.adk.telemetry.setup import maybe_set_otel_providers

# Resolve credentials and project ID explicitly
credentials, project_id = google.auth.default()
print(f"Project ID: {project_id}")

# Build OTel resource with the project ID so Cloud Trace knows where to export
resource = google_cloud.get_gcp_resource(project_id=project_id)

# Get GCP exporters configuration
hooks = google_cloud.get_gcp_exporters(
    enable_cloud_tracing=True,
    google_auth=(credentials, project_id),
)

# Initialize and set global OTel providers
maybe_set_otel_providers(otel_hooks_to_setup=[hooks], otel_resource=resource)

print("✅ Cloud Trace initialized")

Project ID: dev-mq-tech-transfer
✅ Cloud Trace initialized


In [3]:
# Cell 4
from google.adk.agents import Agent

root_agent = Agent(
    name="capital_agent",
    model="gemini-2.5-flash",
    description="Answers geography questions.",
    instruction="Answer the user's question directly and concisely.",
)

print("✅ Agent created")

✅ Agent created


In [4]:
import asyncio
from google.adk.runners import InMemoryRunner
from google.adk.runners import Runner

from google.genai.types import Content, Part
import asyncio
import os

from google.genai import types
from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts.in_memory_artifact_service import InMemoryArtifactService # Optional
from google.adk.planners import BasePlanner, BuiltInPlanner, PlanReActPlanner
from google.adk.models import LlmRequest

from google.genai.types import ThinkingConfig
from google.genai.types import GenerateContentConfig


APP_NAME = "capital_city_app"
USER_ID = "prakash.kc"

In [5]:
from asyncio import events


# session_service = InMemorySessionService()
# session = session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
# runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

# runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)

# session = await runner.session_service.create_session(
#     app_name=APP_NAME,
#     user_id=USER_ID
# )

# user_message = Content(parts=[Part(text="What is the capital of India?")])

# events = runner.run(
#         user_id=USER_ID,
#         session_id=session.id,
#         new_message=user_message,
#     )

# for i in events:
#     print(i)

In [6]:
# Cell 5
import asyncio
from google.adk.runners import InMemoryRunner
from google.genai.types import Content, Part

async def ask(question: str):
    runner = InMemoryRunner(agent=root_agent, app_name="capital_app")

    session = await runner.session_service.create_session(
        app_name="capital_app",
        user_id="user_01"
    )

    user_message = Content(parts=[Part(text=question)])

    async for event in runner.run_async(
        user_id="user_01",
        session_id=session.id,
        new_message=user_message,
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print("Agent:", part.text)



In [7]:
await ask("What is the capital of France?")

Agent: Paris


In [8]:
await ask("What is the capital of India?")

Agent: New Delhi


## Without ADK

In [9]:
# import google.auth
# from google.adk.telemetry import google_cloud
# from google.adk.telemetry.setup import maybe_set_otel_providers
# from google.genai import Client

# # ── Telemetry ──────────────────────────────────────────────
# credentials, project_id = google.auth.default()
# resource = google_cloud.get_gcp_resource(project_id=project_id)
# hooks = google_cloud.get_gcp_exporters(
#     enable_cloud_tracing=True,
#     google_auth=(credentials, project_id),
# )
# maybe_set_otel_providers(otel_hooks_to_setup=[hooks], otel_resource=resource)
# print(f"✅ Cloud Trace initialized for project: {project_id}")

# # ── LLM Client ─────────────────────────────────────────────
# client = Client()

# def ask(question: str) -> str:
#     response = client.models.generate_content(
#         model="gemini-2.5-flash",
#         contents=question,
#     )
#     return response.text

# # ── Ask ────────────────────────────────────────────────────
# print(ask("What is the capital of France?"))
# print(ask("What is the capital of India?"))